<a href="https://colab.research.google.com/github/Saliyah-53/saliyah-stanceeval2026/blob/main/SaliAI_ALLaM_CoT_Vote_Track2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SaliAI — Improving ALLaM on Track 2: CoT + Self-Consistency

We apply the ALLaM experiments (direct answer + chain-of-thought + 5-vote
majority) to Track 2 (Electric Cars / Trimester).
Reference: Fanar 0.828 | previous ALLaM (simple) 0.682. We check whether CoT + vote
lifts ALLaM.

Input: `test_unseen.csv`. Requires a T4 GPU. Run all.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.8 MB/s eta 0:00:00


In [ ]:
import torch, os
assert torch.cuda.is_available(), "فعّل GPU"
print("GPU:", torch.cuda.get_device_name(0))
DATA_DIR="./data"; OUT="./cot_outputs"; os.makedirs(OUT, exist_ok=True)
TEXT_COL="text"; TARGET_COL="target"
# أوصاف أهداف Track 2
TARGET_DESC={"Ecars":"السيارات الكهربائية والتحول إليها","Trimester":"نظام الفصول الدراسية الثلاثة في التعليم"}
def desc(t):
    t=str(t).strip()
    return TARGET_DESC.get(t,t)


GPU: Tesla T4


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pandas as pd, re, zipfile
from collections import Counter
from tqdm.auto import tqdm
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
MID="humain-ai/ALLaM-7B-Instruct-preview"; TAG="allam"
tok=AutoTokenizer.from_pretrained(MID, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
mdl=AutoModelForCausalLM.from_pretrained(MID, quantization_config=bnb, device_map="auto", trust_remote_code=True)
mdl.eval()
print(TAG, "جاهز")

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.23MB            

tokenizer.model: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

allam جاهز


In [ ]:

df=pd.read_csv(f"{DATA_DIR}/test_unseen.csv", keep_default_na=False)
if "tweet_text" in df.columns and TEXT_COL not in df.columns: df=df.rename(columns={"tweet_text":TEXT_COL})
print("اختبار Track 1:", len(df))

def gen(messages, do_sample=False, temp=0.7, max_new=6):
    p=tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    e=tok(p, return_tensors="pt", return_token_type_ids=False).to(mdl.device)
    with torch.no_grad():
        o=mdl.generate(**e, max_new_tokens=max_new, do_sample=do_sample,
                       temperature=temp if do_sample else None, top_p=0.95 if do_sample else None,
                       pad_token_id=tok.eos_token_id)
    return tok.decode(o[0][e["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

def parse(o):
    o=o.strip().lower()
    # نبحث عن آخر كلمة موقف في النص (مهم للـ CoT حيث الجواب في النهاية)
    if "against" in o.split()[-3:] if o.split() else False: pass
    if "against" in o: last_against=o.rfind("against")
    if re.search(r"\bagainst\b", o) and not re.search(r"\bfavor\b", o): return "Against"
    # ترتيب حسب آخر ظهور
    poss=[(o.rfind("against"),"Against"),(o.rfind("favor"),"Favor"),(o.rfind("none"),"None"),
          (o.rfind("معارض"),"Against"),(o.rfind("مؤيد"),"Favor"),(o.rfind("محايد"),"None")]
    poss=[(i,l) for i,l in poss if i>=0]
    if poss: return max(poss)[1]
    return "None"

اختبار Track 1: 644


In [ ]:
#prompts
SYS_DIRECT=("أنت مصنّف مواقف عربي دقيق. أجب بكلمة واحدة فقط: Favor أو Against أو None. "
            "Favor إذا كان مؤيداً للهدف، Against إذا كان معارضاً، None إذا لم يظهر موقف واضح.")
FEW=[("التطعيم أنقذ ملايين الأرواح ولازم الكل ياخذه","لقاح كورونا","Favor"),
     ("ما أثق باللقاح وله أضرار كثيرة","لقاح كورونا","Against"),
     ("متى تفتح مراكز التطعيم؟","لقاح كورونا","None")]

def direct(text,target):
    m=[{"role":"system","content":SYS_DIRECT}]
    for tw,tg,l in FEW:
        m+=[{"role":"user","content":f"الهدف: {tg}\nالتغريدة: {tw}\nالموقف:"},{"role":"assistant","content":l}]
    m.append({"role":"user","content":f"الهدف: {target} ({desc(target)})\nالتغريدة: {text}\nالموقف:"})
    return parse(gen(m, do_sample=False, max_new=6))

SYS_COT=("أنت محلّل مواقف عربي خبير. حلّل التغريدة خطوة بخطوة تجاه الهدف: "
         "(1) هل النص ساخر أو يقتبس رأياً؟ (2) ما نبرة الكاتب الحقيقية؟ (3) هل يؤيد أم يعارض أم لا موقف؟ "
         "ثم اكتب في السطر الأخير فقط: الموقف = Favor أو Against أو None.")
def cot(text,target, do_sample=False, temp=0.7):
    m=[{"role":"system","content":SYS_COT},
       {"role":"user","content":f"الهدف: {target} ({desc(target)})\nالتغريدة: {text}\nحلّل ثم اكتب الموقف في السطر الأخير:"}]
    return parse(gen(m, do_sample=do_sample, temp=temp, max_new=180))

In [ ]:
# voting 5
def save_sub(preds, name):
    txt=f"{OUT}/{name}.txt"; open(txt,"w",encoding="utf-8").write("\n".join(preds)+"\n")
    with zipfile.ZipFile(f"{OUT}/{name}.zip","w",zipfile.ZIP_DEFLATED) as z: z.write(txt, arcname="submission_seen.txt")
    print(f"{name}: {Counter(preds)}")

N_VOTE=5
pd_direct, pd_cot1, pd_cotvote = [], [], []
for _,r in tqdm(df.iterrows(), total=len(df), desc=f"{TAG} reasoning"):
    t=str(r[TEXT_COL]); tg=str(r[TARGET_COL])
    pd_direct.append(direct(t,tg))
    pd_cot1.append(cot(t,tg, do_sample=False))
    votes=[cot(t,tg, do_sample=True, temp=0.7) for _ in range(N_VOTE)]
    pd_cotvote.append(Counter(votes).most_common(1)[0][0])

save_sub(pd_direct,  f"sub_{TAG}_t2_direct")
save_sub(pd_cot1,    f"sub_{TAG}_t2_cot")
save_sub(pd_cotvote, f"sub_{TAG}_t2_cot_vote")
print(">>> ارفع الثلاثة على Track 1 وقارن <<<")

allam reasoning:   0%|          | 0/644 [00:00<?, ?it/s]

sub_allam_t2_direct: Counter({'Favor': 250, 'None': 199, 'Against': 195})
sub_allam_t2_cot: Counter({'Against': 380, 'None': 136, 'Favor': 128})
sub_allam_t2_cot_vote: Counter({'Against': 306, 'Favor': 184, 'None': 154})
>>> ارفع الثلاثة على Track 1 وقارن <<<
